# 실습 2 · 텍스트 — 임베딩·제로샷·군집
한국외국어대학교 국제경영학과 대학원 · 딥러닝 세미나. 데이터: KLUE-YNAT(연합뉴스 제목 7토픽).
**한 줄 목표** — 한국어 텍스트를 튜닝 없이 (a) API 임베딩 → 선형 분류기, (b) LLM 제로샷 라벨, (c) 라벨 없는 군집으로 숫자 변수로 바꾸고, 세 결과를 같은 표에 놓는다.
Colab T4 기준 전체 15분 이내. Cloudflare 무료 한도(하루 10,000 neurons) 중 이 노트북은 약 2,500 neurons 를 쓴다(임베딩 190 · 제로샷 200건 720 · 퓨샷 A/B 120건 1,450 · 군집 이름 20).

## ① 설치·토큰
**관찰 포인트** — 토큰은 코드에 적지 않고 `getpass` 로 넣는다. 계정 ID 는 대시보드 Workers & Pages 화면 오른쪽에 있다.

In [ ]:
!pip -q install umap-learn hdbscan pyarrow requests scikit-learn matplotlib
import os, getpass
os.environ['CLOUDFLARE_ACCOUNT_ID'] = input('Cloudflare Account ID: ').strip()
os.environ['CLOUDFLARE_API_TOKEN']  = getpass.getpass('Cloudflare API Token (Workers AI 권한): ')

### Cloudflare Workers AI 호출 함수
강의 페이지 §3 과 같은 코드다. `USED` 에 neurons 가 누적된다.

In [ ]:
import json, time, requests
ACC = os.environ['CLOUDFLARE_ACCOUNT_ID']; TOK = os.environ['CLOUDFLARE_API_TOKEN']
BASE = f"https://api.cloudflare.com/client/v4/accounts/{ACC}/ai/run/"
USED = {'neurons': 0.0, 'calls': 0}

def _post(model, body, retries=4):
    for k in range(retries):
        r = requests.post(BASE + model, headers={"Authorization": f"Bearer {TOK}"},
                          data=json.dumps(body, ensure_ascii=False).encode('utf-8'), timeout=120).json()
        if r.get('success'):
            USED['calls'] += 1                                   # LLM 은 usage.neurons, 임베딩은 meta.neurons
            USED['neurons'] += float(((r['result'].get('usage') or r['result'].get('meta') or {}).get('neurons') or 0))
            return r['result']
        time.sleep(3 * (k + 1))          # 속도 제한·용량 부족이면 잠시 기다렸다 재시도
    raise RuntimeError(r.get('errors'))

def report(): return f"API 호출 {USED['calls']}회, 사용 neurons {USED['neurons']:.1f} (무료 한도 10,000/일)"

In [ ]:
def embed(texts, model="@cf/baai/bge-m3", batch=100):
    """문장 리스트 → 1024차원 벡터 리스트. 한 번에 100문장."""
    out = []
    for i in range(0, len(texts), batch):
        out.extend(_post(model, {"text": list(texts[i:i+batch])})['data'])
    return out

def zeroshot(text, labels, system, examples=None, model="@cf/meta/llama-3.3-70b-instruct-fp8-fast"):
    """라벨 목록 중 하나를 답하게 한다. examples=[(text,label),...] 를 주면 퓨샷."""
    msgs = [{"role": "system", "content": system}]
    for t, l in (examples or []): msgs += [{"role": "user", "content": t}, {"role": "assistant", "content": l}]
    msgs.append({"role": "user", "content": text})
    ans = (_post(model, {"messages": msgs, "max_tokens": 12, "temperature": 0}).get('response') or '').strip()
    hit = [(ans.find(l), l) for l in labels if l in ans]      # 답 안에 가장 먼저 나오는 라벨
    return (min(hit)[1] if hit else None), ans

## ② 데이터 로드
Hugging Face 의 parquet 을 직접 읽는다. 라벨 순서는 `klue/klue` ynat 의 ClassLabel 순서다(0 IT과학 … 6 정치).
**관찰 포인트** — train 과 val 의 라벨 분포가 다르다(val 은 사회 기사가 40%). macro-F1 을 쓰는 이유다.

In [ ]:
import pandas as pd, numpy as np
U = "https://huggingface.co/api/datasets/klue/klue/parquet/ynat/{}/0.parquet"
tr = pd.read_parquet(U.format('train'));  va = pd.read_parquet(U.format('validation'))
LABELS = ['IT과학','경제','사회','생활문화','세계','스포츠','정치']          # ClassLabel.names 순서
print(tr.shape, va.shape); print(tr[['title','label']].head(3).to_string())
print(pd.DataFrame({'train': tr.label.value_counts(normalize=True).sort_index().round(3),
                    'val':   va.label.value_counts(normalize=True).sort_index().round(3)}).set_axis(LABELS))

## ③ 분할·샘플
YNAT 은 train/val 이 이미 나뉘어 있다. API 비용을 위해 train 3,000 · val 3,000 을 **층화 추출**한다. 자기 데이터에서는 `train_test_split(..., stratify=y)` 한 줄이 이 자리에 온다.

In [ ]:
from sklearn.model_selection import train_test_split
def strat(df, n, seed=0):
    idx, _ = train_test_split(np.arange(len(df)), train_size=n, stratify=df.label.values, random_state=seed); return df.iloc[np.sort(idx)]
tr_s = strat(tr, 3000); va_s = strat(va, 3000)
Xtr_txt, ytr = tr_s.title.tolist(), tr_s.label.values
Xva_txt, yva = va_s.title.tolist(), va_s.label.values
print(len(Xtr_txt), len(Xva_txt))

## ④ 베이스라인 — TF-IDF + 로지스틱 회귀
**한 줄 목표** — 딥러닝 없이 도달하는 점수를 먼저 적는다. 한국어 제목은 띄어쓰기가 불규칙해 **문자 n-gram(1~3)** 이 단어 n-gram 보다 낫다(둘 다 돌려 확인).
**관찰 포인트** — train 3,000 건으로 macro-F1 이 얼마인가. 이 숫자를 아래 임베딩 결과와 비교한다.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix
def tfidf_lr(xtr, ytr, xte, analyzer='char_wb', ngram=(1,3), C=10):
    vec = TfidfVectorizer(analyzer=analyzer, ngram_range=ngram, min_df=2, sublinear_tf=True, token_pattern=r'(?u)\b\w+\b')
    clf = LogisticRegression(C=C, max_iter=3000).fit(vec.fit_transform(xtr), ytr)
    return clf.predict(vec.transform(xte))
for name, kw in [('문자 1~3', dict(analyzer='char_wb', ngram=(1,3))), ('단어 1~2', dict(analyzer='word', ngram=(1,2)))]:
    p = tfidf_lr(Xtr_txt, ytr, Xva_txt, **kw)
    print(f"TF-IDF {name}: macro-F1 {f1_score(yva, p, average='macro'):.3f}  acc {accuracy_score(yva, p):.3f}")
# → 문자 n-gram 이 단어 n-gram 보다 몇 점 높다 (train 3,000 기준 0.7대 초반)

## ⑤ 딥러닝 특징 — bge-m3 임베딩 + 로지스틱 회귀
**한 줄 목표** — 6,000 문장을 1024차원 벡터로 바꾸는 데 neurons 가 얼마나 드는지, 그 벡터 위의 선형 분류기가 TF-IDF 를 이기는지 본다.
**관찰 포인트** — `report()` 의 neurons. 약 190 이면 정상이다(제목 1건 ≈ 29 토큰, 6,000건 ≈ 17만 토큰 × 1,075/M).

In [ ]:
t0 = time.time()
Etr = np.array(embed(Xtr_txt), dtype=np.float32)     # (3000, 1024)
Eva = np.array(embed(Xva_txt), dtype=np.float32)     # (3000, 1024)
print(Etr.shape, Eva.shape, f"{time.time()-t0:.0f}s", report())
np.save('emb_train.npy', Etr); np.save('emb_val.npy', Eva)          # 캐시: 다시 부르지 않는다

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
best = max(((cross_val_score(LogisticRegression(C=C, max_iter=3000), Etr, ytr, cv=StratifiedKFold(5, shuffle=True, random_state=0),
                              scoring='f1_macro').mean(), C) for C in [0.3, 1, 3, 10, 30]))
C_best = best[1]; print('C(5-fold) =', C_best, f'cv macro-F1 {best[0]:.3f}')
clf = LogisticRegression(C=C_best, max_iter=3000).fit(Etr, ytr); p_emb = clf.predict(Eva)
print(f"임베딩+LR: macro-F1 {f1_score(yva, p_emb, average='macro'):.3f}  acc {accuracy_score(yva, p_emb):.3f}")
# → 같은 train 3,000 의 TF-IDF 보다 macro-F1 이 약 0.09 높다 (본 실험 0.779 vs 0.691)

### 제로샷 — llama-3.3-70b 에 라벨만 답하게 한다
**한 줄 목표** — 학습 데이터 0건으로 얻는 점수와 **라벨 미매칭 비율**을 본다. 200건 ≈ 600 neurons.
**관찰 포인트** — 모델이 라벨 외의 말을 붙이는 비율(`None`). 그리고 퓨샷 예시 7개를 다른 7개로 바꾸면 예측이 몇 %나 바뀌는가(아래 셀).

In [ ]:
SYS = "다음은 한국어 뉴스 제목이다. 주제를 " + "/".join(LABELS) + " 중 정확히 하나로 분류하고, 그 라벨만 한 단어로 답하라. 설명은 쓰지 않는다."
zs_df = strat(va_s, 200)                                  # val 샘플 안에서 다시 층화 200건
zs = [zeroshot(t, LABELS, SYS) for t in zs_df.title]
pred = np.array([LABELS.index(l) if l else -1 for l, _ in zs]); y = zs_df.label.values
ok = pred >= 0
print(f"제로샷 200건: macro-F1 {f1_score(y[ok], pred[ok], average='macro'):.3f}  acc {accuracy_score(y, pred):.3f}  미매칭 {(~ok).mean():.1%}", report())
print([a for (_, a), o in zip(zs, ok) if not o][:5])     # 매칭 실패한 원문 답

In [ ]:
# 퓨샷 A/B: 클래스당 1건씩 7개 예시를 두 세트 만들어 같은 60건에 적용 → 두 세트 간 예측 불일치 비율
def pick_examples(seed):
    rng = np.random.default_rng(seed)
    return [(tr.title.iloc[rng.choice(np.where(tr.label.values == c)[0])], LABELS[c]) for c in range(7)]
sub = zs_df.head(60)
pa = [zeroshot(t, LABELS, SYS, pick_examples(11))[0] for t in sub.title]
pb = [zeroshot(t, LABELS, SYS, pick_examples(22))[0] for t in sub.title]
print(f"퓨샷 A acc {np.mean([LABELS[l]==a for l,a in zip(sub.label, pa)]):.3f}  B acc {np.mean([LABELS[l]==b for l,b in zip(sub.label, pb)]):.3f}  A≠B {np.mean([a!=b for a,b in zip(pa,pb)]):.1%}", report())
# → 예시만 바꿔도 예측의 수 % 가 뒤집힌다. 제로샷 라벨을 변수로 쓸 때 이 값을 측정오차로 보고한다

## ⑥ 평가 — seed 5회, 혼동행렬
학습 샘플을 바꿔(3,000 중 2,500 재추출) 5회 반복해 평균±표준편차를 만든다. 표에 쓰는 값은 이것이다.

In [ ]:
rows = []
for s in range(5):
    sub, _ = train_test_split(np.arange(3000), train_size=2500, stratify=ytr, random_state=s)
    pe = LogisticRegression(C=C_best, max_iter=3000).fit(Etr[sub], ytr[sub]).predict(Eva)
    pt = tfidf_lr([Xtr_txt[i] for i in sub], ytr[sub], Xva_txt)
    rows.append({'seed': s, 'TF-IDF+LR': f1_score(yva, pt, average='macro'), '임베딩+LR': f1_score(yva, pe, average='macro')})
res = pd.DataFrame(rows).set_index('seed'); print(res.agg(['mean', 'std']).round(3))
cm = confusion_matrix(yva, p_emb); pairs = sorted(((cm[i,j]+cm[j,i], LABELS[i], LABELS[j]) for i in range(7) for j in range(i+1,7)), reverse=True)[:3]
print('가장 헷갈리는 쌍:', pairs)

## ⑦ 군집·시각화 — UMAP 지도 + K-means/HDBSCAN
**한 줄 목표** — 라벨을 쓰지 않고 임베딩만으로 7개 덩어리가 나오는지, 정답과 ARI 로 얼마나 맞는지 본다.
**관찰 포인트** — 군집은 **1024차원 원 임베딩**에서 하고 UMAP 2차원은 그림에만 쓴다. UMAP 좌표 위 거리는 해석하지 않는다.

In [ ]:
import umap, hdbscan, matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score as ARI, normalized_mutual_info_score as NMI
U = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=0).fit_transform(Eva)
km = [KMeans(7, n_init=10, random_state=s).fit_predict(Eva) for s in range(5)]
ari = [ARI(yva, k) for k in km]; print(f"K-means k=7: ARI {np.mean(ari):.3f}±{np.std(ari, ddof=1):.3f}  NMI {NMI(yva, km[0]):.3f}")
hd = hdbscan.HDBSCAN(min_cluster_size=50).fit_predict(Eva.astype(np.float64)); m = hd >= 0
print(f"HDBSCAN: 군집 {hd.max()+1}개, 잡음 {(~m).mean():.1%}, ARI(잡음 제외) {ARI(yva[m], hd[m]):.3f}")
fig, ax = plt.subplots(1, 2, figsize=(11, 5))
for a, c, t in [(ax[0], yva, 'color = true topic'), (ax[1], km[0], 'color = K-means cluster')]:
    a.scatter(U[:,0], U[:,1], c=c, cmap='tab10', s=4, alpha=.6); a.set_title(t); a.set_xticks([]); a.set_yticks([])
plt.show()

In [ ]:
# 군집 해석: 중심 최근접 제목 5개 → LLM 한 줄 이름 (7회 호출, ~20 neurons)
ct = pd.crosstab(km[0], yva); ct.columns = LABELS
centers = KMeans(7, n_init=10, random_state=0).fit(Eva).cluster_centers_
for c in range(7):
    d = np.linalg.norm(Eva - centers[c], axis=1); d[km[0] != c] = np.inf; reps = [Xva_txt[i] for i in np.argsort(d)[:5]]
    name = _post("@cf/meta/llama-3.3-70b-instruct-fp8-fast", {"messages": [
        {"role": "system", "content": "아래 뉴스 제목 묶음에 공통되는 주제를 한국어 명사구 10자 이내로 한 줄만 답하라."},
        {"role": "user", "content": "\n".join('- ' + r for r in reps)}], "max_tokens": 20, "temperature": 0})['response'].strip()
    print(f"군집 {c} ({(km[0]==c).sum()}건) → {name} | 최다 정답 {ct.columns[ct.values[c].argmax()]} {ct.values[c].max()/ct.values[c].sum():.0%}")
print(report())

## ⑧ 보고표 생성
논문 본문 표(성능)와 부록 표(재현성)를 바로 붙일 수 있는 형태로 만든다.

In [ ]:
import datetime
main_tbl = pd.DataFrame({
  '방법': ['TF-IDF(문자 1~3)+LR', 'bge-m3 임베딩+LR', 'llama-3.3-70b 제로샷'],
  '학습 표본': ['2,500 (seed 5회)', '2,500 (seed 5회)', '0'],
  'macro-F1': [f"{res['TF-IDF+LR'].mean():.3f} ± {res['TF-IDF+LR'].std():.3f}", f"{res['임베딩+LR'].mean():.3f} ± {res['임베딩+LR'].std():.3f}",
               f"{f1_score(y[ok], pred[ok], average='macro'):.3f} (200건, 미매칭 {(~ok).mean():.1%})"]})
appendix = pd.DataFrame([
  ['임베딩 모델', '@cf/baai/bge-m3 (Cloudflare Workers AI), 1024차원, 정규화 없음'],
  ['제로샷 모델', '@cf/meta/llama-3.3-70b-instruct-fp8-fast, temperature 0, max_tokens 12'],
  ['분류기', f'LogisticRegression(C={C_best}, lbfgs, max_iter 3000), C 는 train 5-fold 로 선택'],
  ['TF-IDF', 'char_wb (1,3), min_df 2, sublinear_tf; LogisticRegression C=10'],
  ['군집', 'UMAP(n_neighbors 15, min_dist 0.1, seed 0) 그림용 / K-means k=7, HDBSCAN min_cluster_size 50 는 1024차원'],
  ['seed', '0~4 (학습 재추출)'], ['실행일', datetime.date.today().isoformat()], ['API 사용량', report()]], columns=['항목', '값'])
print(main_tbl.to_markdown(index=False)); print(); print(appendix.to_markdown(index=False))

## ⑨ 자기 데이터로 바꾸기
바꿀 것은 **② 데이터 로드 셀 하나**다. `text` 와 `label` 열을 가진 CSV 를 만들고 아래 셀로 ②를 대체하면 ③~⑧ 이 그대로 돌아간다.
- 라벨이 있으면: 분류·제로샷·군집 전부 실행.
- 라벨이 없으면(공시·의사록·리뷰): `label` 열을 비워 두고 ⑤ 임베딩 → ⑦ 군집 → 군집 이름까지만 실행. 군집 비율·제로샷 라벨이 곧 변수다.
- 긴 문서(수천 자)는 bge-m3 문맥 한도(8,192 토큰) 안에서 문단 단위로 잘라 평균하거나 첫 문단만 쓴다.

In [ ]:
# ② 대체 셀 — text / label 열 이름만 맞추면 된다
# from google.colab import files; up = files.upload(); path = next(iter(up))
path = 'my_data.csv'
df = pd.read_csv(path)
df = df.rename(columns={'text': 'title'}).dropna(subset=['title'])
LABELS = sorted(df.label.dropna().unique().tolist()) if 'label' in df else []   # 라벨 이름 그대로 쓴다(제로샷 프롬프트에도 들어간다)
if LABELS: df['label'] = df.label.map({l: i for i, l in enumerate(LABELS)})
tr, va = train_test_split(df, test_size=0.3, stratify=df.label if LABELS else None, random_state=0)
print(tr.shape, va.shape, LABELS)